# FL Initializer — Run Once Before Round 1
**FL-Pneumonia-Detection | Distributed Setup**

Creates the initial global model (pretrained EfficientNet-B0 with Opacus-compatible
GroupNorm layers) and uploads it to WandB Artifacts as `global-model:round-0`.

**Run this once.** After this, run hospital notebooks → aggregator for each round.

In [ ]:
!pip install opacus==1.4.0 wandb --quiet
print('✅ Packages ready')

In [ ]:
import os, io
import torch
import torch.nn as nn
from torchvision import models
from opacus.validators import ModuleValidator
import wandb
from kaggle_secrets import UserSecretsClient

WORK_DIR   = '/kaggle/working'
WANDB_PROJECT = 'fl-pneumonia-detection'
ARTIFACT_NAME = 'global-model'

wandb_key = UserSecretsClient().get_secret('WANDB_API_KEY')
wandb.login(key=wandb_key)
print('✅ WandB logged in')

In [ ]:
def create_model():
    """EfficientNet-B0 with GroupNorm (Opacus-compatible)."""
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    num_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(num_features, 2),
    )
    model = ModuleValidator.fix(model)  # BatchNorm → GroupNorm
    return model

print('Creating initial global model (pretrained EfficientNet-B0)...')
model = create_model()

errors = ModuleValidator.validate(model, strict=False)
print(f'Opacus validation: {"PASS ✅" if not errors else errors}')

# Smoke-test
x = torch.randn(2, 3, 224, 224)
print(f'Forward pass: {list(x.shape)} → {list(model(x).shape)} ✅')

In [ ]:
# Save model checkpoint
checkpoint_path = f'{WORK_DIR}/global_model_round_0.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'round':            0,
    'description':      'Initial global model — pretrained EfficientNet-B0 (GroupNorm)',
    'classes':          ['NORMAL', 'PNEUMONIA'],
}, checkpoint_path)
print(f'✅ Checkpoint saved: {checkpoint_path}')

# Upload to WandB as artifact tagged round-0
run = wandb.init(
    project=WANDB_PROJECT,
    name='fl-initializer-round-0',
    job_type='initialize',
)

artifact = wandb.Artifact(
    name=ARTIFACT_NAME,
    type='model',
    description='Global FL model checkpoint',
    metadata={'round': 0, 'model': 'EfficientNet-B0-GroupNorm'},
)
artifact.add_file(checkpoint_path, name='global_model.pth')
run.log_artifact(artifact, aliases=['round-0', 'latest'])

wandb.finish()
print(f'✅ Uploaded to WandB as "{ARTIFACT_NAME}:round-0"')
print('\nNow run each hospital notebook (ROUND_NUM=1), then the aggregator.')